# Drivers Graphs

This notebook is made to fetch driver telemetry data in a single track in a single year

In [ ]:
YEAR = 2024

In [ ]:
from fastf1.ergast import Ergast
import fastf1
import os
import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)

ergast = Ergast()
DRIVERS = ergast.get_driver_info(YEAR)["driverId"].to_list()

schedule = fastf1.get_event_schedule(YEAR)
events = schedule["EventName"].to_list()
events = [event for event in events if "Testing" not in event]
races = [idx+1 for idx, _ in enumerate(events)]

for race in races:
    for driver in DRIVERS:
        session = fastf1.get_session(YEAR, race, "R")
        session.load(laps=True, telemetry=True, weather=False)
        driver_slug = f"{driver}_{YEAR}"
        for _, row in session.results.iterrows():
            if row["DriverId"] == driver:
                driver_abv = row["Abbreviation"]
                break
        driver_laps = session.laps.pick_drivers(driver_abv)
        if driver_laps.empty:
            continue
        fastest_lap = driver_laps.pick_fastest()
        if fastest_lap is None:
            fastest_lap = driver_laps.iloc[0]
        start_td = fastest_lap["Time"] - fastest_lap["LapTime"]
        end_td = fastest_lap["Time"]
        telemetry = fastest_lap.get_telemetry()
        telemetry['RelativeSeconds'] = (telemetry['SessionTime'] - start_td).dt.total_seconds()
        telemetry['Compound'] = fastest_lap['Compound']

        position_points = []
        for row in telemetry.itertuples():
            position_points.append((
                round(row.RelativeSeconds,1), 
                round(row.X,1), 
                round(row.Y,1),
                round(row.Z,1),
                round(row.RPM,1),
                round(row.Speed,1), 
                row.nGear, 
                round(row.Throttle,1), 
                row.Brake, 
                row.DRS, 
                row.Status,
                row.Compound
            ))

        os.makedirs(f'data/{YEAR}', exist_ok=True)
        filename = f'data/{YEAR}/telemetry_{driver}_{YEAR}_{race}.csv'

        with open(filename, 'w') as f:
            f.write("seconds,x,y,z,rpm,speed,gear,throttle,brake,drs,status,compound\n")
            for point in position_points:
                f.write(','.join(map(str, point)) + '\n')
                

        print(f"Exported points for {driver} ({DRIVERS.index(driver)+1}/{len(DRIVERS)}) in race {race} ({races.index(race)+1}/{len(races)})")

Exported points for albon (1/25) in race 1 (1/22)
Exported points for alonso (2/25) in race 1 (1/22)
Exported points for antonelli (3/25) in race 1 (1/22)
Exported points for bearman (4/25) in race 1 (1/22)
Exported points for bottas (5/25) in race 1 (1/22)
Exported points for colapinto (6/25) in race 1 (1/22)
Exported points for doohan (7/25) in race 1 (1/22)
Exported points for gasly (8/25) in race 1 (1/22)
Exported points for hamilton (9/25) in race 1 (1/22)
Exported points for hulkenberg (10/25) in race 1 (1/22)
Exported points for lawson (11/25) in race 1 (1/22)
Exported points for leclerc (12/25) in race 1 (1/22)
Exported points for kevin_magnussen (13/25) in race 1 (1/22)
Exported points for norris (14/25) in race 1 (1/22)
Exported points for ocon (15/25) in race 1 (1/22)
Exported points for perez (17/25) in race 1 (1/22)
Exported points for ricciardo (18/25) in race 1 (1/22)
Exported points for russell (19/25) in race 1 (1/22)
Exported points for sainz (20/25) in race 1 (1/22)


DataNotLoadedError: The data you are trying to access has not been loaded yet. See `Session.load`

In [ ]:
# import fastf1
# import logging
# logging.getLogger('fastf1').setLevel(logging.ERROR)
# session = fastf1.get_session(2024, 'Bahrain', 'R')
# session.load()
# [r["DriverId"] for _, r in session.results.iterrows()]